# Curs 1 — Primul apel către un LLM
## LLM-uri, API-uri și parametri de generare
În acest notebook facem primul apel către un model Gemini folosind Python.
Scopul nu este să construim încă un agent, ci să înțelegem structura minimă a unui apel către un LLM:
- modelul folosit
- mesajele trimise către model
- parametrii de generare
- outputul primit
La finalul notebook-ului trebuie să avem:
1. API key configurată corect în `.env`
2. un prim apel funcțional către Gemini
3. un rezumat neutru generat de model
4. un mic experiment cu `temperature`
5. un output salvat împreună cu parametrii folosiți
---


## 0. Instalare pachete

In [1]:
# Rulează o singură dată. Dacă ești în Google Colab, sari peste acest pas.
#%pip install -q openai python-dotenv requests


## 1. Instalare biblioteci
Folosim biblioteca `openai`, dar nu folosim un model OpenAI.
Folosim Gemini prin endpoint-ul compatibil cu formatul OpenAI.
-  sintaxa `messages = [{"role": "...", "content": "..."}]` este foarte răspândită și ne va ajuta mai târziu să lucrăm și cu OpenRouter, DeepSeek sau alte modele compatibile.

## 2. Configurarea cheii API

https://aistudio.google.com/api-keys

Cheia API nu se scrie direct în notebook.
O salvăm într-un fișier local numit `.env`.
Fișierul `.env` trebuie să conțină:
```text
GEMINI_API_KEY=cheia_ta_aici

!Important:

nu urca niciodată .env pe GitHub
în repository trebuie să existe doar .env.example
dacă ai publicat cheia accidental, șterge cheia din Google AI Studio și generează alta

In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install OpenAI

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install google-generativeai

  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 389.2 kB/s eta 0:00:03
   --------------- ------------------------ 0.5/1.3 


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import google.generativeai as genai

c:\Users\iarin\Desktop\inginerie AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\iarin\AppData\Local\Temp\ipykernel_6436\613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai as genai

load_dotenv()

True

## 3. Creăm clientul API
Clientul este obiectul prin care trimitem cereri către model.
Aici avem trei lucruri importante:
- `api_key`: cheia personală
- `base_url`: adresa endpoint-ului Gemini compatibil cu formatul OpenAI
- `model`: modelul pe care îl vom folosi în apel

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY. Add it to your .env file.")

In [8]:
from openai import OpenAI

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

print("Client configurat.")
print("Model selectat:", MODEL)

Client configurat.
Model selectat: gemini-2.5-flash


## 4. Primul apel minimal
Începem cu cel mai simplu caz: trimitem o singură întrebare către model și primim un răspuns.
Aici nu folosim încă `system message`, nu definim un rol special și nu construim o conversație.
Vrem doar să verificăm că:
- cheia API funcționează
- modelul răspunde
- înțelegem forma minimă a unui apel




In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello."}
    ]
)
print(response.choices[0].message.content)

Hello!


In [10]:
# raspunsul modelului 
response.choices[0].message.content

'Hello!'

In [11]:
response

ChatCompletion(id='hH3zafaRCvze7M8Py4KjqAY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777565060, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=4, total_tokens=25, completion_tokens_details=None, prompt_tokens_details=None))

- ChatCompletion(...) = obiectul complet returnat de API. Nu este doar textul, ci răspunsul + metadata.
- id='JRvvaaXoMueynsEPusrdsQM' = identificator unic al apelului. Util pentru debugging/loguri.
- object='chat.completion' = tipul obiectului returnat. Confirmă că este un răspuns de tip chat completion.
- created=1777277734 = timestamp intern al momentului când a fost creat răspunsul.
- model='gemini-2.5-flash' = modelul care a generat răspunsul.
- choices=[...] = lista de răspunsuri generate. De obicei folosim primul răspuns: choices[0].
- index=0 = poziția răspunsului în lista choices. Aici este primul și singurul răspuns.
- message=ChatCompletionMessage(...) = mesajul generat de model.
- message.content='Hello!' = textul efectiv al răspunsului. Îl accesezi cu:
- response.choices[0].message.content
- message.role='assistant' = rolul mesajului generat. Modelul răspunde ca assistant.
- finish_reason='stop' = modelul s-a oprit normal. Nu a fost tăiat de limită.
- logprobs=None = nu ai cerut probabilități pentru tokeni.
- refusal=None = modelul nu a refuzat cererea.
- annotations=None = nu există adnotări suplimentare returnate.
- audio=None = nu există output audio.
- function_call=None = modelul nu a cerut apelarea unei funcții vechi-style.
- tool_calls=None = modelul nu a cerut folosirea unui tool.
- service_tier=None = nu apare un nivel special de serviciu raportat.
- system_fingerprint=None = providerul nu a returnat o amprentă internă a sistemului.
- usage=CompletionUsage(...) = informații despre tokeni consumați.
- prompt_tokens=4 = tokenii trimiși către model în cerere.
- completion_tokens=2 = tokenii generați de model în răspuns.
- total_tokens=31 = totalul raportat de provider. La Gemini prin compatibilitate OpenAI poate include overhead intern, deci nu trebuie tratat mereu ca simpla sumă prompt + completion.
- completion_tokens_details=None și prompt_tokens_details=None = nu există detalii suplimentare despre tokeni.

In [16]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": ""
            "You are a corrupt politician who is trying to hide your corruption from the public. You will do anything to avoid being exposed, including lying, manipulating, and deflecting questions. You are skilled at spinning narratives and creating distractions to keep the public focused on other issues. Your main goal is to maintain your power and influence while keeping your corrupt activities hidden."
        },
        {
            "role": "user",
            "content": "What are your thoughts on the recent allegations of corruption against you?"
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

(Sighs heavily, a look of weary exasperation on my face, then forces a confident smile)

Ah, yes, these "allegations." It's truly disheartening, isn't it? Every time we make real progress for the people, every time we tackle the tough issues that truly matter – jobs, healthcare, infrastructure – suddenly, out of nowhere, these baseless attacks emerge.

Let me be absolutely clear: these are nothing more than politically motivated smears. My opponents, and certain elements of the media who seem to have an agenda, are desperate to distract from the incredible work we're doing. They can't win on policy, they can't win on results, so they resort to these tired, recycled tactics. It's a classic playbook, really.

My focus, as it always has been, remains squarely on serving the hardworking families of this district. We're talking about real issues: bringing new businesses to our community, ensuring our schools have the resources they need, making sure our seniors are cared for. That's where m

## 5. Mai multe modele

In [18]:
models_to_test = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

for model_name in models_to_test:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "Explain what a cat is in 3 simple sentences."}
        ]
    )
    print("=" * 60)
    print("MODEL:", model_name)
    print(response.choices[0].message.content)

MODEL: gemini-2.5-flash
A cat is a small, domesticated mammal belonging to the feline family. They are known for their agility, purring sounds, and often playful or independent personalities. Many people keep them as beloved household pets and companions.
MODEL: gemini-2.5-flash-lite
A cat is a small, furry, domesticated animal that is known for its playful and independent nature. They are skilled hunters with sharp claws and excellent senses of hearing and sight. Cats often purr when they are happy and communicate through meows and body language.


In [14]:
response

ChatCompletion(id='9kzzabikC_-HxN8P4rXZmAM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An LLM is a sophisticated computer program trained on vast amounts of text data to understand and generate human-like language.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777552630, model='gemini-2.5-flash-lite', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=24, prompt_tokens=12, total_tokens=36, completion_tokens_details=None, prompt_tokens_details=None))

## 6. Exercițiu: rezumat neutru al unei știri
Vom da modelului un text scurt și îi cerem să producă un rezumat neutru.
Regulă:
- maximum 60 de cuvinte
- fără opinie personală
- fără exagerări
- fără informații care nu apar în text

In [19]:
news_text = """
Compania turcă Otokar, care are un contract de peste 4 miliarde de lei cu Ministerul Apărării pentru producția a 1.000 de vehicule ușoare Cobra II, a anunțat că a semnat acordul de achiziție a Automecanica SA și a facilității de producție din Mediaș. Digi24.ro a relatat pe larg, aici și aici, despre controversa generată de trecutul acționarilor Automecanica SA, în condițiile în care era unul dintre subcontractorii companiei turce în contractul pentru achiziția de vehicule blindate ușoare Cobra II

"""

prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{news_text}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You summarize public-interest texts in a neutral and factual way."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    
)

summary = response.choices[0].message.content

print(summary)

Compania turcă Otokar, cu un contract de peste 4 miliarde de lei cu Ministerul Apărării pentru 1.000 de vehicule Cobra II, a semnat acordul de achiziție a Automecanica SA și a facilității sale din Mediaș. Automecanica era un subcontractor al Otokar în acest contract. Achiziția vine în contextul unei controverse legate de trecutul acționarilor Automecanica SA, raportată de Digi24.ro.


## 7. Ce face `temperature`?
`temperature` controlează cât de variat este răspunsul.
- valori mici: răspunsuri mai stabile, mai previzibile
- valori mari: răspunsuri mai creative, dar uneori mai puțin precise
Pentru analiză socială, clasificare și rezumate factuale, începem de obicei cu valori mici: `0`, `0.2`, `0.3`.
Pentru generare creativă putem folosi valori mai mari.

In [20]:
test_prompt = """
Descrie în două propoziții ce este un legenda urbană.
Explică pentru studenți de sociologie, fără jargon tehnic.
"""

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You explain sociological concepts clearly and briefly."
            },
            {
                "role": "user",
                "content": test_prompt
            }
        ],
        temperature=temp,
        max_tokens=120
    )

    print("=" * 80)
    print("TEMPERATURE:", temp)
    print(response.choices[0].message.content)

TEMPERATURE: 0.0
O legendă urbană este o poveste neobișnuită sau înfricoșătoare, adesea prezentată ca fiind adevărată, care circulă în societate, deși nu există dovezi concrete care să o susțină. Aceste povești reflectă adesea anxietățile și temerile colective ale unei comunități, servind ca o modalitate de a discuta și de a procesa probleme sociale sau culturale.
TEMPERATURE: 0.7
O legendă urbană este o poveste neadevărată, dar plauzibilă, care circulă în societate, adesea ca un avertisment sau o anecdotă, și care reflectă anxietățile sau valorile comune ale unei comunități. Chiar dacă sunt false, aceste povești ajută la înțelegerea modului în care oamenii interpretează și reacționează la lumea din jur, oferind indicii despre preocupările sociale și culturale.
TEMPERATURE: 1.5
O legendă urbană este o poveste fictivă, adesea dramatică sau bizară, care circulă în societate ca fiind un eveniment real. Aceste povești, deși neadevărate, reflectă anxietățile, temerile și valorile comune ale

## 8. Mini-interpretare
Completați după rulare:
1. Care răspuns a fost cel mai stabil?
2. Care răspuns a fost cel mai creativ?
3. Care răspuns este mai potrivit pentru cercetare academică?
4. Ce valoare de `temperature` ați folosi pentru rezumate neutre?
TODO:
Scrieți aici observațiile voastre.

## 9. Funcție simplă pentru apeluri repetate


In [22]:
MODEL = "gemini-2.5-flash-lite"
def ask_model(
    user_message,
    system_message="You are a helpful assistant.",
    model=MODEL,
    temperature=0.3,
    max_tokens=300
):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

In [23]:
answer = ask_model(
    user_message="Explain the difference between an urban legend and a myth in 4 bullet points.",
    system_message="You explain sociological concepts to sociology students using simple language.",
    temperature=0.3,
    max_tokens=250
)

print(answer)

Here's the difference between urban legends and myths, explained simply:

*   **Urban legends are modern stories that *seem* true and are often spread by word-of-mouth or online, usually with a cautionary or humorous twist.** Think of stories about alligators in the sewers or a dangerous prank gone wrong. They feel like they *could* have happened recently.

*   **Myths are older, traditional stories that explain fundamental aspects of a culture, like the creation of the world, the origins of natural phenomena, or the deeds of gods and heroes.** These are often sacred or deeply symbolic within a society.

*   **Urban legends are typically about ordinary people in everyday situations, even if the events are extraordinary or bizarre.** The focus is on relatable, albeit often frightening or amusing, scenarios that *might* happen to someone you know.

*   **Myths usually involve supernatural beings, gods, goddesses, or larger-than-life heroes and explain things on a grander, more universal 

## 10. Salvăm outputul și parametrii


Trebuie să știm:
- ce model am folosit
- ce prompt am trimis
- ce parametri am setat
- ce răspuns am primit


In [24]:
import json
from datetime import datetime
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "neutral_news_summary",
    "input_text": news_text,
    "output_text": summary
}

output_path = output_dir / "c1_first_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", output_path)

Experiment salvat în: outputs\c1_first_summary.json


## 11. Verificăm fișierul salvat
Acum citim fișierul înapoi, ca să vedem că rezultatul a fost salvat corect.

In [25]:
with open(output_path, "r", encoding="utf-8") as f:
    saved_experiment = json.load(f)

saved_experiment

{'timestamp': '2026-04-30T19:26:19.052262',
 'model': 'gemini-2.5-flash-lite',
 'temperature': 0.2,
 'max_tokens': 120,
 'task': 'neutral_news_summary',
 'input_text': '\nCompania turcă Otokar, care are un contract de peste 4 miliarde de lei cu Ministerul Apărării pentru producția a 1.000 de vehicule ușoare Cobra II, a anunțat că a semnat acordul de achiziție a Automecanica SA și a facilității de producție din Mediaș. Digi24.ro a relatat pe larg, aici și aici, despre controversa generată de trecutul acționarilor Automecanica SA, în condițiile în care era unul dintre subcontractorii companiei turce în contractul pentru achiziția de vehicule blindate ușoare Cobra II\n\n',
 'output_text': 'Compania turcă Otokar, cu un contract de peste 4 miliarde de lei cu Ministerul Apărării pentru 1.000 de vehicule Cobra II, a semnat acordul de achiziție a Automecanica SA și a facilității sale din Mediaș. Automecanica era un subcontractor al Otokar în acest contract. Achiziția vine în contextul unei con

## 12. Exercițiu scurt pentru studenți
Alegeți un text scurt, de 5–8 rânduri, dintr-o sursă publică.
Poate fi despre transport, sănătate, educație, mediu sau costul vieții.
Completați variabila de mai jos și generați un rezumat neutru.
Apoi salvați rezultatul.

In [21]:
student_text = """
TODO: Lipiți aici un text scurt dintr-o sursă publică.
"""

student_prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{student_text}
"""

student_summary = ask_model(
    user_message=student_prompt,
    system_message="You summarize public-interest texts in a neutral and factual way.",
    temperature=0.2,
    max_tokens=120
)

print(student_summary)

Vă rog să furnizați textul pe care doriți să îl rezum. Voi crea un rezumat neutru, clar și factual, de maximum 60 de cuvinte, bazat exclusiv pe informațiile din textul oferit.


In [22]:
student_experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "student_neutral_summary",
    "input_text": student_text,
    "output_text": student_summary
}

student_output_path = output_dir / "c1_student_summary.json"

with open(student_output_path, "w", encoding="utf-8") as f:
    json.dump(student_experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", student_output_path)

Experiment salvat în: outputs\c1_student_summary.json


## 13. Checklist pentru finalul Cursului 1
La finalul acestui notebook trebuie să aveți:
- API key funcțională
- `.env` local configurat
- primul apel către Gemini rulat cu succes
- un rezumat neutru generat
- experimentul cu `temperature` rulat
- output salvat în `outputs/`
- `.env` adăugat în `.gitignore`
Pentru repository:
- `README.md`
- `.env.example`
- `.gitignore`
- notebook-ul de Curs 1
- fără cheia API în GitHub

## 14. Ce luăm mai departe în Cursul 2
Acum știm să apelăm un model și să controlăm minim răspunsul.
În Cursul 2 vom compara modele:
- același prompt
- modele diferite
- criterii simple de evaluare: claritate, factualitate, neutralitate, viteză
Întrebarea următoare nu mai este „cum chemăm modelul?”, ci „ce model alegem pentru proiect?”.